# treefi Sample Usage

This notebook shows how to use `treefi` with:

- scikit-learn regression
- scikit-learn classification
- XGBoost sklearn-compatible wrappers
- CatBoost sklearn-compatible wrappers

The examples focus on `feature_interactions(...)`, `feature_importance(...)`, and `summarize_model(...)`.

In [2]:
import numpy as np
import xgboost as xgb
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

import treefi


## Data Setup

In [6]:
# set np random seed for reproducibility
np.random.seed(42)

diabetes = load_diabetes(as_frame=True)
X_reg = (diabetes
    .frame[diabetes.feature_names]
    .assign(rand_1=lambda df: np.random.normal(size=len(df)),
    rand_2=lambda df: np.random.normal(size=len(df)),
    rand_3=lambda df: np.random.normal(size=len(df))
    )
)
y_reg = diabetes.frame[diabetes.target.name]

cancer = load_breast_cancer(as_frame=True)
X_clf = (cancer
    .frame[cancer.feature_names]
    .assign(rand_1=lambda df: np.random.normal(size=len(df)),
    rand_2=lambda df: np.random.normal(size=len(df)),
    rand_3=lambda df: np.random.normal(size=len(df))
    )
)
y_clf = cancer.frame[cancer.target.name]

X_reg


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,rand_1,rand_2,rand_3
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,0.496714,-2.067442,0.681891
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,-0.138264,-0.089120,1.846707
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,0.647689,-1.304470,0.583928
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,1.523030,0.669673,-0.359292
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,-0.234153,0.366598,0.590655
...,...,...,...,...,...,...,...,...,...,...,...,...,...
437,0.041708,0.050680,0.019662,0.059744,-0.005697,-0.002566,-0.028674,-0.002592,0.031193,0.007207,-1.380101,1.188393,0.529693
438,-0.005515,0.050680,-0.015906,-0.067642,0.049341,0.079165,-0.028674,0.034309,-0.018114,0.044485,-1.703382,2.526932,-0.070499
439,0.041708,0.050680,-0.015906,0.017293,-0.037344,-0.013840,-0.024993,-0.011080,-0.046883,0.015491,-0.055548,-0.530869,0.486502
440,-0.045472,-0.044642,0.039062,0.001215,0.016318,0.015283,-0.028674,0.026560,0.044529,-0.025930,0.384065,-0.489439,0.064474


## 1. scikit-learn Regression Example

In [ ]:
rf_reg = RandomForestRegressor(n_estimators=10, max_depth=4, random_state=0)
rf_reg.fit(X_reg, y_reg)

reg_interactions = treefi.feature_interactions(
    rf_reg,
    max_interaction_depth=1,
    sort_by="gain",
    top_k=10,
)
(reg_interactions
    .sort_values("gain", ascending=False)
)

,interaction,interaction_order,gain,cover,fscore,weighted_fscore,average_weighted_fscore,average_gain,expected_gain,average_tree_index,...,rank_gain,rank_fscore,rank_expected_gain,rank_consensus,backend,model_type,occurrence_count,tree_count,feature_count,path_probability_sum
1,bmi|bp,2.0,1.997174e+07,14104.0,19,3.868778,0.193439,1.085654e+06,4.085509e+06,4.142857,...,1.714286,5.142857,4.428571,3.761905,sklearn,RandomForestRegressor,19,7.0,2.0,3.868778
2,bmi|s5,2.0,1.962292e+07,12558.0,18,7.459276,0.418238,1.088716e+06,8.027763e+06,4.555556,...,2.444444,5.222222,3.333333,3.666667,sklearn,RandomForestRegressor,18,9.0,2.0,7.459276
18,bp|s5,2.0,6.960984e+06,4795.0,6,1.504525,0.240385,1.180489e+06,1.751182e+06,6.000000,...,5.000000,11.750000,5.250000,7.333333,sklearn,RandomForestRegressor,6,4.0,2.0,1.504525
38,age|s5,2.0,4.721484e+06,3475.0,4,1.314480,0.345400,1.134836e+06,1.543068e+06,6.333333,...,6.333333,6.000000,4.000000,5.444444,sklearn,RandomForestRegressor,4,3.0,2.0,1.314480
16,bmi|rand_2,2.0,4.174544e+06,3286.0,4,0.690045,0.172511,1.043636e+06,6.990449e+05,4.000000,...,6.250000,12.250000,7.750000,8.750000,sklearn,RandomForestRegressor,4,4.0,2.0,0.690045
24,rand_2|s5,2.0,3.901399e+06,3406.0,4,1.020362,0.255090,9.753498e+05,9.723067e+05,5.250000,...,9.250000,9.500000,6.750000,8.500000,sklearn,RandomForestRegressor,4,4.0,2.0,1.020362
0,bmi|rand_1,2.0,3.784107e+06,2399.0,3,0.484163,0.125566,1.235724e+06,6.298321e+05,1.500000,...,3.500000,10.000000,7.500000,7.000000,sklearn,RandomForestRegressor,3,2.0,2.0,0.484163
39,s5,1.0,3.741182e+06,3057.0,11,3.841629,0.361551,3.394505e+05,1.597951e+06,6.666667,...,4.666667,1.666667,4.666667,3.666667,sklearn,RandomForestRegressor,11,3.0,1.0,3.841629
43,s1|s5,2.0,3.578104e+06,2342.0,3,0.314480,0.104827,1.192701e+06,3.858691e+05,7.333333,...,7.000000,10.333333,9.666667,9.000000,sklearn,RandomForestRegressor,3,3.0,2.0,0.314480
37,age|s1,2.0,3.560884e+06,2608.0,3,0.500000,0.166667,1.186961e+06,5.365784e+05,6.333333,...,6.000000,8.333333,7.666667,7.333333,sklearn,RandomForestRegressor,3,3.0,2.0,0.500000


In [10]:
reg_importance = treefi.feature_importance(rf_reg, sort_by="gain", top_k=10)
(reg_importance
    .sort_values("gain", ascending=False)
)

,feature,gain,cover,weight,total_gain,total_cover,weighted_fscore,average_weighted_fscore,expected_gain,average_tree_index,average_tree_depth,backend,model_type,occurrence_count,tree_count
0,bmi,244955.330242,184.709677,31,7.593615e+06,5726.0,12.954751,0.417895,6.419462e+06,4.000000,0.419355,sklearn,RandomForestRegressor,31,10.0
1,s5,224560.360637,194.818182,22,4.940328e+06,4286.0,9.696833,0.440765,3.724797e+06,5.136364,0.863636,sklearn,RandomForestRegressor,22,10.0
2,s4,91192.693338,144.333333,3,2.735781e+05,433.0,0.979638,0.326546,1.066722e+05,3.000000,2.333333,sklearn,RandomForestRegressor,3,3.0
3,bp,86711.133262,104.277778,18,1.560800e+06,1877.0,4.246606,0.235923,4.707171e+05,4.944444,2.444444,sklearn,RandomForestRegressor,18,10.0
4,s6,78870.510484,67.375000,8,6.309641e+05,539.0,1.219457,0.152432,1.635512e+05,5.125000,2.500000,sklearn,RandomForestRegressor,8,5.0
5,rand_1,50907.393145,67.875000,8,4.072591e+05,543.0,1.228507,0.153563,7.411787e+04,3.000000,2.625000,sklearn,RandomForestRegressor,8,5.0
6,s2,45549.193101,107.800000,5,2.277460e+05,539.0,1.219457,0.243891,5.573338e+04,2.600000,2.600000,sklearn,RandomForestRegressor,5,5.0
7,age,39408.640264,96.416667,12,4.729037e+05,1157.0,2.617647,0.218137,1.097166e+05,5.333333,2.333333,sklearn,RandomForestRegressor,12,6.0
8,s3,37262.007496,92.833333,6,2.235720e+05,557.0,1.260181,0.210030,6.240544e+04,3.666667,2.500000,sklearn,RandomForestRegressor,6,6.0
9,rand_3,32757.130313,55.111111,9,2.948142e+05,496.0,1.122172,0.124686,3.360135e+04,4.666667,2.777778,sklearn,RandomForestRegressor,9,7.0


## 2. scikit-learn Classification Example

In [12]:
rf_clf = RandomForestClassifier(n_estimators=10, max_depth=4, random_state=0)
rf_clf.fit(X_clf, y_clf)

clf_interactions = treefi.feature_interactions(
    rf_clf,
    max_interaction_depth=1,
    sort_by="gain",
    top_k=10,
)
(clf_interactions
.sort_values("gain", ascending=False)
)

,interaction,interaction_order,gain,cover,fscore,weighted_fscore,average_weighted_fscore,average_gain,expected_gain,average_tree_index,...,rank_gain,rank_fscore,rank_expected_gain,rank_consensus,backend,model_type,occurrence_count,tree_count,feature_count,path_probability_sum
27,mean perimeter|worst concave points,2.0,607.856937,3640.0,4,0.448155,0.174868,166.671415,79.159633,3.5,...,5.50,10.5,7.0,7.666667,sklearn,RandomForestClassifier,4,2.0,2.0,0.448155
7,worst radius,1.0,592.181142,3721.0,7,4.323374,0.721954,109.555410,481.604014,3.0,...,7.75,2.0,3.5,4.416667,sklearn,RandomForestClassifier,7,4.0,1.0,4.323374
34,worst radius|worst texture,2.0,582.068901,2742.0,3,0.460457,0.193761,193.573478,88.853215,5.0,...,5.50,7.5,5.5,6.166667,sklearn,RandomForestClassifier,3,2.0,2.0,0.460457
63,mean concave points|mean fractal dimension,2.0,535.700099,2547.0,3,0.738137,0.246046,178.566700,131.542771,7.0,...,1.00,2.0,2.0,1.666667,sklearn,RandomForestClassifier,3,1.0,2.0,0.738137
14,worst concave points|worst fractal dimension,2.0,429.003521,2617.0,2,0.734622,0.367311,214.501760,158.632184,2.0,...,4.00,7.0,5.0,5.333333,sklearn,RandomForestClassifier,2,2.0,2.0,0.734622
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,mean concave points|mean texture,2.0,112.957154,897.0,1,0.576450,0.576450,112.957154,65.114142,6.0,...,10.00,3.0,1.0,4.666667,sklearn,RandomForestClassifier,1,1.0,2.0,0.576450
76,worst area|worst concavity,2.0,86.707232,977.0,1,0.347979,0.347979,86.707232,30.172288,8.0,...,9.00,8.0,7.0,8.000000,sklearn,RandomForestClassifier,1,1.0,2.0,0.347979
77,worst area|worst smoothness,2.0,84.252065,791.0,1,0.021090,0.021090,84.252065,1.776845,8.0,...,10.00,11.0,15.0,12.000000,sklearn,RandomForestClassifier,1,1.0,2.0,0.021090
8,worst concave points,1.0,16.900585,38.0,1,0.066784,0.066784,16.900585,1.128686,0.0,...,9.00,11.0,12.0,10.666667,sklearn,RandomForestClassifier,1,1.0,1.0,0.066784


## 3. XGBoost sklearn API Examples

In [13]:
xgb_reg = xgb.XGBRegressor(
    objective="reg:squarederror",
    max_depth=3,
    n_estimators=10,
    learning_rate=0.1,
    random_state=0,
)
xgb_reg.fit(X_reg, y_reg)

xgb_reg_interactions = treefi.feature_interactions(
    xgb_reg,
    max_interaction_depth=1,
    sort_by="gain",
    top_k=10,
)
(xgb_reg_interactions
.sort_values("gain", ascending=False)
)

,interaction,interaction_order,gain,cover,fscore,weighted_fscore,average_weighted_fscore,average_gain,expected_gain,average_tree_index,...,rank_gain,rank_fscore,rank_expected_gain,rank_consensus,backend,model_type,occurrence_count,tree_count,feature_count,path_probability_sum
0,bmi|s5,2.0,9.599988e+06,12831.0,18,9.346154,0.551395,474136.710525,4.647884e+06,4.500000,...,1.900000,2.900000,1.500000,2.100000,xgboost,XGBRegressor,18,10.0,2.0,9.346154
1,bmi,1.0,5.671140e+06,8298.0,26,11.131222,0.483861,158101.434804,2.029501e+06,4.500000,...,5.100000,2.100000,4.900000,4.033333,xgboost,XGBRegressor,26,10.0,1.0,11.131222
5,bmi|bp,2.0,3.946619e+06,6109.0,8,1.348416,0.167582,448736.689953,7.089492e+05,5.142857,...,2.142857,6.142857,5.571429,4.619048,xgboost,XGBRegressor,8,7.0,2.0,1.348416
3,s5,1.0,3.762572e+06,4932.0,13,11.158371,0.907919,302276.371240,3.721383e+06,4.500000,...,5.700000,2.800000,1.600000,3.366667,xgboost,XGBRegressor,13,10.0,1.0,11.158371
7,bmi|s2,2.0,2.720809e+06,4659.0,6,1.522624,0.250905,493416.109210,6.675183e+05,4.200000,...,3.200000,5.200000,4.000000,4.133333,xgboost,XGBRegressor,6,5.0,2.0,1.522624
2,bmi|s3,2.0,1.783106e+06,2633.0,3,1.273756,0.424585,594368.513000,7.371136e+05,3.000000,...,3.000000,4.000000,3.666667,3.555556,xgboost,XGBRegressor,3,3.0,2.0,1.273756
14,bp|s5,2.0,1.460488e+06,3011.0,5,1.812217,0.362443,292097.600240,5.147759e+05,6.400000,...,5.800000,6.600000,4.000000,5.466667,xgboost,XGBRegressor,5,5.0,2.0,1.812217
12,bp|s6,2.0,1.370996e+06,2750.0,4,0.873303,0.218326,342749.065615,3.294861e+05,6.000000,...,4.000000,8.750000,5.500000,6.083333,xgboost,XGBRegressor,4,4.0,2.0,0.873303
13,bp|s2,2.0,1.114342e+06,1871.0,3,0.214932,0.071644,371447.247757,7.987860e+04,5.000000,...,5.000000,10.666667,8.000000,7.888889,xgboost,XGBRegressor,3,3.0,2.0,0.214932
11,bmi|rand_3,2.0,8.912419e+05,1886.0,2,0.954751,0.477376,445620.938200,4.276375e+05,5.000000,...,2.000000,5.000000,3.000000,3.333333,xgboost,XGBRegressor,2,2.0,2.0,0.954751


In [14]:
xgb_clf = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    max_depth=3,
    n_estimators=10,
    learning_rate=0.1,
    random_state=0,
)
xgb_clf.fit(X_clf, y_clf)

xgb_clf_interactions = treefi.feature_interactions(
    xgb_clf,
    max_interaction_depth=1,
    sort_by="gain",
    top_k=10,
)
(xgb_clf_interactions
.sort_values("gain", ascending=False)
)

,interaction,interaction_order,gain,cover,fscore,weighted_fscore,average_weighted_fscore,average_gain,expected_gain,average_tree_index,...,rank_gain,rank_fscore,rank_expected_gain,rank_consensus,backend,model_type,occurrence_count,tree_count,feature_count,path_probability_sum
3,mean concave points|worst texture,2.0,1467.913235,1047.987024,5,1.195455,0.242894,315.517299,333.099546,1.750000,...,2.750000,7.500000,5.500000,5.250000,xgboost,XGBClassifier,5,4.0,2.0,1.195455
12,worst concave points|worst perimeter,2.0,1145.327869,946.523891,5,2.254506,0.491779,236.638632,540.362307,3.500000,...,4.000000,3.500000,2.500000,3.333333,xgboost,XGBClassifier,5,4.0,2.0,2.254506
0,worst concave points|worst texture,2.0,928.888032,643.806645,3,0.268796,0.089599,309.629344,81.388127,2.666667,...,1.000000,6.333333,6.000000,4.444444,xgboost,XGBClassifier,3,3.0,2.0,0.268796
1,radius error|worst concave points,2.0,893.904957,814.353726,3,1.633694,0.544565,297.968319,496.277242,2.666667,...,2.000000,4.333333,3.000000,3.111111,xgboost,XGBClassifier,3,3.0,2.0,1.633694
2,worst concave points|worst radius,2.0,884.934109,610.480195,3,1.902491,0.634164,294.978036,568.888925,2.666667,...,3.000000,2.333333,2.000000,2.444444,xgboost,XGBClassifier,3,3.0,2.0,1.902491
13,worst perimeter,1.0,870.020083,487.902283,5,4.022536,0.877817,194.655760,869.816358,3.500000,...,6.000000,1.500000,1.000000,2.833333,xgboost,XGBClassifier,5,4.0,1.0,4.022536
7,worst concave points,1.0,849.650696,1274.640003,12,6.989705,0.590345,50.528740,487.599283,3.750000,...,5.625000,3.500000,5.500000,4.875000,xgboost,XGBClassifier,12,8.0,1.0,6.989705
11,mean concave points|worst perimeter,2.0,824.455780,544.827705,3,1.287068,0.429023,274.818593,351.136735,2.333333,...,4.000000,6.000000,4.333333,4.777778,xgboost,XGBClassifier,3,3.0,2.0,1.287068
6,worst radius,1.0,785.782944,481.747685,5,4.070826,0.814165,157.156589,773.877520,4.600000,...,7.400000,2.000000,3.000000,4.133333,xgboost,XGBClassifier,5,5.0,1.0,4.070826
10,concave points error|mean concave points,2.0,644.252245,386.490478,2,0.121489,0.060745,322.126122,39.103741,1.500000,...,2.000000,7.000000,6.000000,5.000000,xgboost,XGBClassifier,2,2.0,2.0,0.121489


## 4. CatBoost sklearn API Examples

In [16]:
cat_reg = CatBoostRegressor(
    depth=3,
    iterations=10,
    learning_rate=0.1,
    verbose=False,
)
cat_reg.fit(X_reg, y_reg)

cat_reg_interactions = treefi.feature_interactions(
    cat_reg,
    max_interaction_depth=1,
    sort_by="interaction",
    top_k=10,
)
(cat_reg_interactions
.sort_values("gain", ascending=False)
)

,interaction,interaction_order,gain,cover,fscore,weighted_fscore,average_weighted_fscore,average_gain,expected_gain,average_tree_index,...,rank_gain,rank_fscore,rank_expected_gain,rank_consensus,backend,model_type,occurrence_count,tree_count,feature_count,path_probability_sum
2,bmi|s5,2.0,30849.848036,7072.0,10,4.0,0.437500,3091.930455,13937.618130,4.000000,...,1.750000,2.500000,1.750000,2.000000,catboost,CatBoostRegressor,10,4.0,2.0,4.0
3,bmi|bp,2.0,29328.067082,4420.0,6,2.0,0.375000,4167.535069,8717.716726,2.500000,...,1.500000,2.000000,1.500000,1.666667,catboost,CatBoostRegressor,6,2.0,2.0,2.0
9,bmi|s4,2.0,21454.509367,5746.0,8,3.0,0.416667,1987.886424,10366.090381,4.500000,...,2.000000,2.000000,2.000000,2.000000,catboost,CatBoostRegressor,8,2.0,2.0,3.0
4,bmi,1.0,14839.587762,3536.0,17,8.0,0.592857,849.084786,8778.758602,4.285714,...,3.714286,3.714286,3.857143,3.761905,catboost,CatBoostRegressor,17,7.0,1.0,8.0
11,bmi|s2,2.0,11805.804757,3094.0,4,1.0,0.250000,2951.451189,3078.111938,3.000000,...,1.000000,1.000000,1.000000,1.000000,catboost,CatBoostRegressor,4,1.0,2.0,1.0
7,bp|rand_3,2.0,9994.424217,3094.0,4,1.0,0.250000,2498.606054,2857.599279,1.000000,...,1.000000,1.000000,1.000000,1.000000,catboost,CatBoostRegressor,4,1.0,2.0,1.0
15,bmi|s3,2.0,8896.312640,3094.0,4,1.0,0.250000,2224.078160,2502.276112,5.000000,...,1.000000,1.000000,1.000000,1.000000,catboost,CatBoostRegressor,4,1.0,2.0,1.0
0,s5,1.0,8630.588403,3094.0,9,7.0,0.857143,940.262979,7890.649007,4.428571,...,4.142857,4.714286,4.142857,4.333333,catboost,CatBoostRegressor,9,7.0,1.0,7.0
10,s2,1.0,5966.805799,1326.0,12,3.0,0.250000,497.233817,2300.756974,5.333333,...,2.666667,2.000000,3.333333,2.666667,catboost,CatBoostRegressor,12,3.0,1.0,3.0
1,bp,1.0,5850.740758,1326.0,7,3.0,0.583333,973.429697,3109.407277,2.000000,...,4.333333,3.666667,4.333333,4.111111,catboost,CatBoostRegressor,7,3.0,1.0,3.0


In [17]:
cat_clf = CatBoostClassifier(
    depth=3,
    iterations=10,
    learning_rate=0.1,
    verbose=False,
)
cat_clf.fit(X_clf, y_clf)

cat_clf_interactions = treefi.feature_interactions(
    cat_clf,
    max_interaction_depth=1,
    sort_by="interaction",
    top_k=10,
)
(cat_clf_interactions
.sort_values("gain", ascending=False)
)

,interaction,interaction_order,gain,cover,fscore,weighted_fscore,average_weighted_fscore,average_gain,expected_gain,average_tree_index,...,rank_gain,rank_fscore,rank_expected_gain,rank_consensus,backend,model_type,occurrence_count,tree_count,feature_count,path_probability_sum
13,texture error|worst concave points,2.0,228.098894,3983.0,4,1.0,0.250000,57.024724,58.324490,2.000000,...,1.000000,1.000000,1.000000,1.000000,catboost,CatBoostClassifier,4,1.0,2.0,1.0
20,worst radius|worst texture,2.0,150.459668,3983.0,4,1.0,0.250000,37.614917,40.902193,4.000000,...,1.000000,1.000000,1.000000,1.000000,catboost,CatBoostClassifier,4,1.0,2.0,1.0
2,mean concavity|worst compactness,2.0,130.628528,3983.0,4,1.0,0.250000,32.657132,44.473667,0.000000,...,1.000000,1.000000,1.000000,1.000000,catboost,CatBoostClassifier,4,1.0,2.0,1.0
23,area error|worst radius,2.0,117.787737,3983.0,4,1.0,0.250000,29.446934,31.751986,5.000000,...,1.000000,1.000000,1.000000,1.000000,catboost,CatBoostClassifier,4,1.0,2.0,1.0
12,texture error|worst radius,2.0,109.743986,1707.0,2,1.0,0.500000,54.871993,54.740691,2.000000,...,2.000000,3.000000,2.000000,2.333333,catboost,CatBoostClassifier,2,1.0,2.0,1.0
29,mean concave points|worst texture,2.0,105.538784,3983.0,4,1.0,0.250000,26.384696,29.092400,7.000000,...,1.000000,1.000000,1.000000,1.000000,catboost,CatBoostClassifier,4,1.0,2.0,1.0
10,worst radius,1.0,87.332540,1707.0,7,3.0,0.583333,23.134395,74.391811,3.666667,...,3.333333,3.666667,3.666667,3.555556,catboost,CatBoostClassifier,7,3.0,1.0,3.0
16,mean perimeter|worst symmetry,2.0,81.257773,3983.0,4,1.0,0.250000,20.314443,26.630237,3.000000,...,1.000000,1.000000,1.000000,1.000000,catboost,CatBoostClassifier,4,1.0,2.0,1.0
21,worst concave points|worst radius,2.0,71.939451,1707.0,2,1.0,0.500000,35.969726,38.584890,4.000000,...,2.000000,3.000000,2.000000,2.333333,catboost,CatBoostClassifier,2,1.0,2.0,1.0
8,area error|radius error,2.0,69.106785,3983.0,4,1.0,0.250000,17.276696,21.821808,1.000000,...,1.000000,1.000000,1.000000,1.000000,catboost,CatBoostClassifier,4,1.0,2.0,1.0


## 5. Grouped Output with `summarize_model(...)`

In [19]:
summary = treefi.summarize_model(xgb_reg, max_interaction_depth=1, top_k=10)

summary.metadata

{'backend': 'xgboost', 'model_type': 'XGBRegressor'}

In [21]:
(summary
.interactions
.sort_values("gain", ascending=False)
)

,interaction,interaction_order,gain,cover,fscore,weighted_fscore,average_weighted_fscore,average_gain,expected_gain,average_tree_index,...,rank_gain,rank_fscore,rank_expected_gain,rank_consensus,backend,model_type,occurrence_count,tree_count,feature_count,path_probability_sum
1,bmi|s5,2.0,9.599988e+06,12831.0,18,9.346154,0.551395,474136.710525,4.647884e+06,4.500000,...,1.900000,2.900000,1.500000,2.100000,xgboost,XGBRegressor,18,10.0,2.0,9.346154
2,bmi,1.0,5.671140e+06,8298.0,26,11.131222,0.483861,158101.434804,2.029501e+06,4.500000,...,5.100000,2.100000,4.900000,4.033333,xgboost,XGBRegressor,26,10.0,1.0,11.131222
7,bmi|bp,2.0,3.946619e+06,6109.0,8,1.348416,0.167582,448736.689953,7.089492e+05,5.142857,...,2.142857,6.142857,5.571429,4.619048,xgboost,XGBRegressor,8,7.0,2.0,1.348416
0,s5,1.0,3.762572e+06,4932.0,13,11.158371,0.907919,302276.371240,3.721383e+06,4.500000,...,5.700000,2.800000,1.600000,3.366667,xgboost,XGBRegressor,13,10.0,1.0,11.158371
5,bmi|s2,2.0,2.720809e+06,4659.0,6,1.522624,0.250905,493416.109210,6.675183e+05,4.200000,...,3.200000,5.200000,4.000000,4.133333,xgboost,XGBRegressor,6,5.0,2.0,1.522624
3,bmi|s3,2.0,1.783106e+06,2633.0,3,1.273756,0.424585,594368.513000,7.371136e+05,3.000000,...,3.000000,4.000000,3.666667,3.555556,xgboost,XGBRegressor,3,3.0,2.0,1.273756
13,bp|s5,2.0,1.460488e+06,3011.0,5,1.812217,0.362443,292097.600240,5.147759e+05,6.400000,...,5.800000,6.600000,4.000000,5.466667,xgboost,XGBRegressor,5,5.0,2.0,1.812217
14,bp|s6,2.0,1.370996e+06,2750.0,4,0.873303,0.218326,342749.065615,3.294861e+05,6.000000,...,4.000000,8.750000,5.500000,6.083333,xgboost,XGBRegressor,4,4.0,2.0,0.873303
11,bmi|rand_3,2.0,8.912419e+05,1886.0,2,0.954751,0.477376,445620.938200,4.276375e+05,5.000000,...,2.000000,5.000000,3.000000,3.333333,xgboost,XGBRegressor,2,2.0,2.0,0.954751
9,bmi|s6,2.0,8.313539e+05,782.0,1,0.262443,0.262443,831353.906500,2.181834e+05,1.000000,...,3.000000,8.000000,5.000000,5.333333,xgboost,XGBRegressor,1,1.0,2.0,0.262443


In [22]:
(summary
.importance
.sort_values("gain", ascending=False)
)

,feature,gain,cover,weight,total_gain,total_cover,weighted_fscore,average_weighted_fscore,expected_gain,average_tree_index,average_tree_depth,backend,model_type,occurrence_count,tree_count
0,s5,289428.623908,379.384615,13,3.762572e+06,4932.0,11.158371,0.858336,3.721383e+06,4.307692,0.000000,xgboost,XGBRegressor,13,10.0
1,bmi,92490.185829,204.000000,21,1.942294e+06,4284.0,9.692308,0.461538,9.669801e+05,3.666667,1.095238,xgboost,XGBRegressor,21,10.0
4,bp,40041.323081,114.000000,12,4.804959e+05,1368.0,3.095023,0.257919,1.405254e+05,5.083333,1.833333,xgboost,XGBRegressor,12,8.0
5,s6,27286.100772,100.400000,5,1.364305e+05,502.0,1.135747,0.227149,3.563936e+04,5.000000,2.000000,xgboost,XGBRegressor,5,5.0
6,rand_3,26611.050800,211.000000,2,5.322210e+04,422.0,0.954751,0.477376,2.547520e+04,5.000000,2.000000,xgboost,XGBRegressor,2,2.0
2,s3,20585.318350,184.750000,4,8.234127e+04,739.0,1.671946,0.417986,3.480646e+04,4.250000,2.000000,xgboost,XGBRegressor,4,4.0
9,rand_1,20115.718760,73.000000,2,4.023144e+04,146.0,0.330317,0.165158,8.868174e+03,7.500000,2.000000,xgboost,XGBRegressor,2,2.0
7,rand_2,18194.960900,52.000000,1,1.819496e+04,52.0,0.117647,0.117647,2.140584e+03,4.000000,2.000000,xgboost,XGBRegressor,1,1.0
8,age,17447.978500,47.000000,1,1.744798e+04,47.0,0.106335,0.106335,1.855328e+03,6.000000,2.000000,xgboost,XGBRegressor,1,1.0
3,s2,15729.029286,85.333333,9,1.415613e+05,768.0,1.737557,0.193062,3.274934e+04,4.888889,2.000000,xgboost,XGBRegressor,9,8.0


In [23]:
summary.leaf_stats

,interaction,leaf_effect_mean,leaf_effect_var
0,s5,-3.726072,0.000000
1,bmi|s5,-3.726072,0.000000
2,bmi,-0.351287,2.437521
3,bmi|s3,-2.923692,0.000000
4,s3,-2.485715,0.000000
5,bmi|s2,2.913548,0.030715
6,s2,2.913548,0.030715
7,bmi|bp,-0.578073,1.253987
8,bp,-0.436457,1.462985
9,bmi|s6,0.140631,0.000000


## 6. Cross-Validated Stability and Noise Diagnostics

The CV helpers let you inspect whether strong interactions or features are stable across folds instead of trusting a single fitted model. This section also shows how injected random columns can surface as suspicious even when they rank non-trivially in a single fitted model.


In [14]:
cv_interactions = treefi.cross_validated_interactions(
    rf_reg,
    X_reg,
    y_reg,
    n_splits=5,
    top_k=10,
)

cv_interactions.metadata

{'task': 'regression', 'splitter': 'KFold', 'n_splits': 5}

In [15]:
cv_interactions.interaction_summary[
    [
        "interaction",
        "mean_gain",
        "fold_presence_rate",
        "selection_rate_top_k",
        "gain_cv",
        "cv_instability_flag",
        "suspicious_feature_score",
    ]
].sort_values("mean_gain", ascending=False).head(10)


,interaction,mean_gain,fold_presence_rate,selection_rate_top_k,rank_stability_score,rare_fold_flag,overfit_suspect_flag
1,bmi|s5,2.963443e+07,1.0,1.0,0.811583,False,False
2,bmi,9.648360e+06,1.0,1.0,0.514641,False,False
0,s5,7.367254e+06,1.0,1.0,0.679748,False,False
3,bmi|bp|s5,6.368014e+06,1.0,1.0,0.205183,False,False
4,bmi|bp,5.309213e+06,0.8,0.8,0.254164,False,False
11,bmi|s3,5.105896e+06,0.6,0.6,0.211166,False,False
43,bmi|s2,4.253000e+06,0.8,0.8,0.284912,False,False
24,age|s5,3.835417e+06,0.8,0.8,0.236412,False,False
61,bp|s5,3.807631e+06,0.6,0.6,0.252351,False,False
10,bmi|s3|s5,3.619078e+06,1.0,1.0,0.239112,False,False


A practical reading pattern is:

- look for high `mean_gain`
- prefer high `fold_presence_rate` and `selection_rate_top_k`
- treat high `gain_cv` or `cv_instability_flag` as a warning for unstable patterns
- use `suspicious_feature_score` as a triage aid, not proof that a feature is useless


In [16]:
xgb_noise = xgb.XGBRegressor(
    objective="reg:squarederror",
    max_depth=4,
    n_estimators=20,
    learning_rate=0.1,
    random_state=0,
)

cv_importance = treefi.cross_validated_importance(
    xgb_noise,
    X_reg_noise,
    y_reg,
    n_splits=3,
    top_k=8,
)

cv_importance.importance_summary[
    [
        "feature",
        "mean_gain",
        "fold_presence_rate",
        "selection_rate_top_k",
        "gain_cv",
        "overfit_suspect_flag",
        "high_weight_low_gain_flag",
        "low_consensus_top_k_flag",
        "suspicious_feature_score",
    ]
].sort_values("mean_gain", ascending=False).head(12)


,feature,mean_gain,mean_expected_gain,fold_presence_rate,selection_rate_top_k,gain_cv
0,s5,216295.296042,5.142402e+06,1.0,1.0,0.151566
1,bmi,168308.410673,3.873903e+06,1.0,1.0,0.120219
2,bp,57110.280267,3.621767e+05,1.0,1.0,0.297713
7,s6,48092.987733,1.785444e+05,1.0,1.0,0.235378
5,s3,39390.711809,1.193631e+05,1.0,1.0,0.234620
9,s4,37719.389254,7.007464e+04,1.0,1.0,0.576184
3,s1,30034.655147,5.142356e+04,1.0,1.0,0.192696
6,s2,25759.219484,6.737236e+04,1.0,1.0,0.403156
4,age,24983.030077,6.045712e+04,1.0,1.0,0.100842
8,sex,21389.415254,1.026805e+04,1.0,1.0,0.473965


A useful follow-up is to inspect just the injected columns. This is often clearer than scanning the full table for `rand_*` rows.


In [ ]:
cv_importance.importance_summary[
    cv_importance.importance_summary["feature"].str.startswith("rand_")
][
    [
        "feature",
        "mean_gain",
        "fold_presence_rate",
        "selection_rate_top_k",
        "high_total_gain_low_density_flag",
        "high_weight_low_gain_flag",
        "weak_signal_density_flag",
        "suspicious_feature_score",
    ]
].sort_values("suspicious_feature_score", ascending=False)
